# **SMOTE leakage experiment**

This notebook tests the three COM725 models (logistic regression, random forest, decision tree) on the HCV data and answers three questions.

1. How well do the models do with and without synthetic patients (SMOTE)?
2. Does a two-step model (healthy or not, then type of liver damage) beat a single four-class model?
3. How much are the scores inflated when SMOTE is applied to the whole dataset before splitting?

Every result uses 5-fold cross-validation repeated five times (25 test folds).

# **Importing Libraries**

In [1]:
# importing necessary libraries / dependencies
import warnings

import numpy as np
import pandas as pd
from imblearn.over_sampling import SMOTENC
from imblearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    balanced_accuracy_score, classification_report, f1_score, recall_score,
)
from sklearn.model_selection import (
    RepeatedStratifiedKFold, StratifiedKFold, cross_val_predict,
    cross_validate,
)
from sklearn.preprocessing import QuantileTransformer
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220, "display.max_columns", 20)

# **Loading the DataFrame**

The raw file is used, not the imputed one, so that missing values are filled inside each training fold. The labels are cleaned the same way as in `cleaning.ipynb`.

In [2]:
# retrieving the raw dataset and dropping the row number column
df = pd.read_csv("hcvdat0.csv").drop(columns=["Unnamed: 0"])

# cleaning the labels and removing the suspected blood donors
df["Category"] = df["Category"].str.replace(r"^[^=]*=", "", regex=True)
df = df[~df["Category"].str.lower().str.contains("suspect")]
df["Category"] = df["Category"].replace({"Blood Donor": "Healthy"})

# encoding sex into numeric values (f = 0, m = 1)
df["Sex"] = (df["Sex"] == "m").astype(int)

# sex stays last (column index 11) as SMOTENC needs its position
NUMERIC = ["Age", "ALB", "ALP", "ALT", "AST", "BIL", "CHE", "CHOL",
           "CREA", "GGT", "PROT"]
FEATURES = NUMERIC + ["Sex"]

In [3]:
# dataset A: healthy vs not healthy
X_a, y_a = df[FEATURES], (df["Category"] != "Healthy").astype(int)

# dataset B: only the 75 liver patients, predicting the type of damage
liver = df[df["Category"] != "Healthy"]
X_b, y_b = liver[FEATURES], liver["Category"]

# checking the size of each dataset
DATASETS = {"A: healthy vs not healthy": (X_a, y_a),
            "B: liver damage type (75)": (X_b, y_b)}
for name, (X, y) in DATASETS.items():
    print(name, len(X), y.value_counts().to_dict())

A: healthy vs not healthy 608 {0: 533, 1: 75}
B: liver damage type (75) 75 {'Cirrhosis': 30, 'Hepatitis': 24, 'Fibrosis': 21}


# **Models**

The three models use the same settings as the COM725 notebook. Each one is built twice: without and with synthetic patients. SMOTE sits inside the pipeline, so synthetic patients are only added to the training folds.

In [4]:
# building the three COM725 models, without and with synthetic patients
MODELS = ["Logistic regression", "Random forest", "Decision tree"]
PIPELINES = {}
for synthetic in [False, True]:

    # missing values are filled with the median of the training fold
    steps = [("impute", SimpleImputer(strategy="median"))]

    # synthetic patients are added to the training folds only
    if synthetic:
        steps.append(("smote", SMOTENC(
            categorical_features=[len(FEATURES) - 1], k_neighbors=5,
            random_state=42)))

    # defining logistic regression pipeline
    PIPELINES[("Logistic regression", synthetic)] = Pipeline(steps + [
        ("quantile", QuantileTransformer(
            output_distribution="normal", random_state=42)),
        ("clf", LogisticRegression(
            max_iter=1000, class_weight="balanced", random_state=42))])

    # defining random forest pipeline
    PIPELINES[("Random forest", synthetic)] = Pipeline(steps + [
        ("clf", RandomForestClassifier(
            n_estimators=200, class_weight="balanced", random_state=42,
            n_jobs=-1))])

    # defining decision tree pipeline
    PIPELINES[("Decision tree", synthetic)] = Pipeline(steps + [
        ("clf", DecisionTreeClassifier(
            class_weight="balanced", random_state=42))])

# **With and without synthetic patients**

In [5]:
# 5-fold cross-validation repeated five times (25 test folds)
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=42)

# testing every model on both datasets, without and with SMOTE
rows = []
for data_name, (X, y) in DATASETS.items():

    # recall and precision of the sick class only exist for two classes
    scoring = {"macro_f1": "f1_macro",
               "balanced_acc": "balanced_accuracy"}
    if y.nunique() == 2:
        scoring.update({"recall_not_healthy": "recall",
                        "precision_not_healthy": "precision"})

    for model_name in MODELS:
        for synthetic in [False, True]:
            s = cross_validate(PIPELINES[(model_name, synthetic)], X, y,
                               cv=cv, scoring=scoring, error_score="raise")

            # mean and standard deviation over the 25 folds
            row = {"dataset": data_name, "model": model_name,
                   "synthetic": synthetic}
            for key in scoring:
                row[key] = round(s["test_" + key].mean(), 3)
                row[key + "_sd"] = round(s["test_" + key].std(), 3)
            rows.append(row)

results = pd.DataFrame(rows)
results

,dataset,model,synthetic,macro_f1,macro_f1_sd,balanced_acc,balanced_acc_sd,recall_not_healthy,recall_not_healthy_sd,precision_not_healthy,precision_not_healthy_sd
0,A: healthy vs not healthy,Logistic regression,False,0.897,0.039,0.935,0.034,0.915,0.064,0.753,0.095
1,A: healthy vs not healthy,Logistic regression,True,0.918,0.026,0.934,0.031,0.896,0.066,0.830,0.076
2,A: healthy vs not healthy,Random forest,False,0.903,0.039,0.859,0.052,0.720,0.103,0.983,0.035
3,A: healthy vs not healthy,Random forest,True,0.944,0.035,0.948,0.037,0.912,0.067,0.893,0.070
4,A: healthy vs not healthy,Decision tree,False,0.858,0.043,0.854,0.055,0.741,0.114,0.781,0.116
5,A: healthy vs not healthy,Decision tree,True,0.881,0.036,0.895,0.049,0.827,0.105,0.775,0.096
6,B: liver damage type (75),Logistic regression,False,0.632,0.108,0.638,0.105,NaN,NaN,NaN,NaN
7,B: liver damage type (75),Logistic regression,True,0.625,0.107,0.634,0.105,NaN,NaN,NaN,NaN
8,B: liver damage type (75),Random forest,False,0.674,0.106,0.684,0.102,NaN,NaN,NaN,NaN
9,B: liver damage type (75),Random forest,True,0.675,0.135,0.686,0.131,NaN,NaN,NaN,NaN


Healthy vs not healthy is an easy task. Without synthetic patients the macro F1 is 0.897 for logistic regression, 0.903 for random forest and 0.858 for the decision tree. SMOTE raises these to 0.918, 0.944 and 0.881. Its main effect is on the random forest, which finds 91% of sick patients with SMOTE against 72% without.

The type of liver damage is much harder. With only 75 patients the macro F1 is between 0.56 and 0.68, and SMOTE makes little difference.

# **Two-step model vs single model**

The two-step model first decides healthy or not healthy, then predicts the type of liver damage for the patients it flags. It is compared with one random forest that predicts all four classes. Three cut-offs are tried for the first step.

In [6]:
# all four classes, on the same 25 splits
CLASSES = ["Healthy", "Hepatitis", "Fibrosis", "Cirrhosis"]
X4, y4 = df[FEATURES], df["Category"]

# cut-offs of step 1 for calling a patient "not healthy"
THRESHOLDS = [0.5, 0.3, 0.2]

records = []
for train, test in cv.split(X4, y4):
    X_tr, X_te = X4.iloc[train], X4.iloc[test]
    y_tr, y_te = y4.iloc[train], y4.iloc[test]
    sick_tr = y_tr != "Healthy"

    # step 1: healthy vs not healthy (random forest + synthetic patients)
    stage1 = clone(PIPELINES[("Random forest", True)])
    stage1.fit(X_tr, sick_tr.astype(int))
    p_sick = stage1.predict_proba(X_te)[:, 1]

    # step 2: type of damage, trained on sick training patients only
    stage2 = clone(PIPELINES[("Random forest", False)])
    stage2.fit(X_tr[sick_tr], y_tr[sick_tr])
    type_pred = stage2.predict(X_te)

    # single model: all four classes at once (random forest + synthetic)
    single = clone(PIPELINES[("Random forest", True)])
    single_pred = single.fit(X_tr, y_tr).predict(X_te)

    # collecting the predictions of the single model and each cut-off
    preds = {"Single-stage RF + SMOTENC": single_pred}
    for t in THRESHOLDS:
        preds[f"Chained (threshold {t})"] = np.where(
            p_sick >= t, type_pred, "Healthy")

    # scoring every set of predictions on this fold
    for name, pred in preds.items():
        row = {"model": name,
               "macro_f1": f1_score(y_te, pred, average="macro"),
               "balanced_acc": balanced_accuracy_score(y_te, pred)}
        for c in CLASSES:
            row["recall_" + c] = recall_score(
                y_te, pred, labels=[c], average="macro", zero_division=0)
        records.append(row)

# mean (standard deviation) over the 25 folds
chain = pd.DataFrame(records)
mean = chain.groupby("model", sort=False).mean().round(3)
sd = chain.groupby("model", sort=False).std().round(3)
(mean.astype(str) + " (" + sd.astype(str) + ")")

,macro_f1,balanced_acc,recall_Healthy,recall_Hepatitis,recall_Fibrosis,recall_Cirrhosis
model,,,,,,
Single-stage RF + SMOTENC,0.659 (0.09),0.663 (0.085),0.988 (0.012),0.484 (0.278),0.368 (0.21),0.813 (0.155)
Chained (threshold 0.5),0.678 (0.074),0.691 (0.069),0.987 (0.011),0.492 (0.242),0.43 (0.245),0.853 (0.13)
Chained (threshold 0.3),0.655 (0.066),0.712 (0.068),0.953 (0.022),0.552 (0.21),0.468 (0.219),0.873 (0.121)
Chained (threshold 0.2),0.623 (0.063),0.718 (0.064),0.909 (0.035),0.6 (0.223),0.488 (0.209),0.873 (0.121)


The two-step model with a cut-off of 0.5 scores slightly higher than the single model (macro F1 0.678 against 0.659), but the difference is smaller than the variation between folds. Lower cut-offs find more sick patients (Fibrosis recall 0.43 to 0.49) but flag more healthy patients (Healthy recall 0.987 to 0.909). Neither design is clearly better.

# **SMOTE leakage demonstration**

The leaky way applies SMOTE to the whole dataset first and cross-validates afterwards, so synthetic copies of the test patients end up in the training folds. The honest way applies SMOTE inside the training folds only. Both are run on the same models.

In [7]:
# adding the four-class problem as dataset C
DATASETS_ALL = dict(DATASETS)
DATASETS_ALL["C: all four classes"] = (X4, y4)
SCORES = {"macro_f1": "f1_macro", "balanced_acc": "balanced_accuracy"}

rows = []
for data_name, (X, y) in DATASETS_ALL.items():

    # leak: impute and oversample using all patients before any split
    X_imp = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(X),
                         columns=FEATURES)
    X_res, y_res = SMOTENC(categorical_features=[len(FEATURES) - 1],
                           k_neighbors=5, random_state=42
                           ).fit_resample(X_imp, y)
    print(data_name, ":", len(X), "rows ->", len(X_res), "after SMOTE")

    for model_name in MODELS:

        # leaky: cross-validate on the already oversampled data
        leaky = cross_validate(PIPELINES[(model_name, False)], X_res, y_res,
                               cv=cv, scoring=SCORES)

        # honest: SMOTE only inside each training fold, real test patients
        honest = cross_validate(PIPELINES[(model_name, True)], X, y,
                                cv=cv, scoring=SCORES)
        rows.append({
            "dataset": data_name, "model": model_name,
            "macro_f1 honest": honest["test_macro_f1"].mean(),
            "macro_f1 LEAKY": leaky["test_macro_f1"].mean(),
            "bal_acc honest": honest["test_balanced_acc"].mean(),
            "bal_acc LEAKY": leaky["test_balanced_acc"].mean()})

# difference between the leaky and honest macro F1
leak = pd.DataFrame(rows)
leak["F1 inflation"] = leak["macro_f1 LEAKY"] - leak["macro_f1 honest"]
leak.round(3)

A: healthy vs not healthy : 608 rows -> 1066 after SMOTE


B: liver damage type (75) : 75 rows -> 90 after SMOTE


C: all four classes : 608 rows -> 2132 after SMOTE


,dataset,model,macro_f1 honest,macro_f1 LEAKY,bal_acc honest,bal_acc LEAKY,F1 inflation
0,A: healthy vs not healthy,Logistic regression,0.918,0.969,0.934,0.969,0.051
1,A: healthy vs not healthy,Random forest,0.944,0.986,0.948,0.986,0.042
2,A: healthy vs not healthy,Decision tree,0.881,0.963,0.895,0.963,0.082
3,B: liver damage type (75),Logistic regression,0.625,0.680,0.634,0.684,0.055
4,B: liver damage type (75),Random forest,0.675,0.737,0.686,0.740,0.062
5,B: liver damage type (75),Decision tree,0.597,0.699,0.609,0.704,0.101
6,C: all four classes,Logistic regression,0.620,0.876,0.672,0.876,0.256
7,C: all four classes,Random forest,0.659,0.994,0.663,0.994,0.335
8,C: all four classes,Decision tree,0.561,0.958,0.586,0.958,0.397


The leak inflates every score. For healthy vs not healthy the inflation is small (0.04 to 0.08 macro F1) because the task is already easy. For the four classes it is very large: the random forest goes from 0.659 to 0.994 and the decision tree from 0.561 to 0.958. Scores close to 99% on this dataset can be produced by the leak alone.

# Classification reports for all four classes

One prediction per patient, from a single 5-fold pass.

In [8]:
# a single 5-fold pass so every patient is predicted once
fold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# leaky data: SMOTE on all patients first (same as the previous cell)
X_imp = pd.DataFrame(SimpleImputer(strategy="median").fit_transform(X4),
                     columns=FEATURES)
X_res, y_res = SMOTENC(categorical_features=[len(FEATURES) - 1],
                       k_neighbors=5, random_state=42
                       ).fit_resample(X_imp, y4)

for model_name in MODELS:

    # honest: real patients only in the test folds, SMOTE inside training
    honest = cross_val_predict(PIPELINES[(model_name, True)], X4, y4,
                               cv=fold)
    print("=" * 70)
    print(model_name, "- HONEST (SMOTE inside training folds), n =", len(y4))
    print(classification_report(y4, honest, labels=CLASSES, digits=3))

    # leaky: cross-validate on the data oversampled before splitting
    leaky = cross_val_predict(PIPELINES[(model_name, False)], X_res, y_res,
                              cv=fold)
    print(model_name, "- LEAKY (SMOTE before splitting), n =", len(y_res))
    print(classification_report(y_res, leaky, labels=CLASSES, digits=3))

Logistic regression - HONEST (SMOTE inside training folds), n = 608
              precision    recall  f1-score   support

     Healthy      0.988     0.944     0.965       533
   Hepatitis      0.359     0.583     0.444        24
    Fibrosis      0.367     0.524     0.431        21
   Cirrhosis      0.733     0.733     0.733        30

    accuracy                          0.905       608
   macro avg      0.612     0.696     0.644       608
weighted avg      0.929     0.905     0.915       608

Logistic regression - LEAKY (SMOTE before splitting), n = 2132
              precision    recall  f1-score   support

     Healthy      0.953     0.944     0.948       533
   Hepatitis      0.818     0.842     0.830       533
    Fibrosis      0.823     0.831     0.827       533
   Cirrhosis      0.934     0.906     0.920       533

    accuracy                          0.881      2132
   macro avg      0.882     0.881     0.881      2132
weighted avg      0.882     0.881     0.881      2132


Random forest - HONEST (SMOTE inside training folds), n = 608
              precision    recall  f1-score   support

     Healthy      0.981     0.992     0.987       533
   Hepatitis      0.619     0.542     0.578        24
    Fibrosis      0.471     0.381     0.421        21
   Cirrhosis      0.806     0.833     0.820        30

    accuracy                          0.946       608
   macro avg      0.719     0.687     0.701       608
weighted avg      0.941     0.946     0.943       608



Random forest - LEAKY (SMOTE before splitting), n = 2132
              precision    recall  f1-score   support

     Healthy      1.000     0.987     0.993       533
   Hepatitis      0.991     1.000     0.995       533
    Fibrosis      0.994     0.998     0.996       533
   Cirrhosis      0.996     0.996     0.996       533

    accuracy                          0.995      2132
   macro avg      0.995     0.995     0.995      2132
weighted avg      0.995     0.995     0.995      2132

Decision tree - HONEST (SMOTE inside training folds), n = 608
              precision    recall  f1-score   support

     Healthy      0.973     0.953     0.963       533
   Hepatitis      0.280     0.292     0.286        24
    Fibrosis      0.304     0.333     0.318        21
   Cirrhosis      0.658     0.833     0.735        30

    accuracy                          0.900       608
   macro avg      0.554     0.603     0.576       608
weighted avg      0.907     0.900     0.903       608



Decision tree - LEAKY (SMOTE before splitting), n = 2132
              precision    recall  f1-score   support

     Healthy      0.979     0.953     0.966       533
   Hepatitis      0.940     0.962     0.951       533
    Fibrosis      0.959     0.961     0.960       533
   Cirrhosis      0.981     0.981     0.981       533

    accuracy                          0.964      2132
   macro avg      0.965     0.964     0.964      2132
weighted avg      0.965     0.964     0.964      2132



The honest reports show where the models really struggle. Hepatitis and Fibrosis recall is between 0.29 and 0.58 for all three models. In the leaky reports the same classes reach 0.83 to 1.00, because the test folds are full of synthetic copies of patients the model has already seen.